## 第 2 周 · 第 1 天练习 —— 用 OpenRouter 免费调用多模型

**作者**：Venkata Narayana reddy Tumu

### 练习目标（理念）

本练习对应第 2 周「多模型 / 第三方路由」相关概念：通过 **OpenRouter** 的 OpenAI 兼容接口，用熟悉的 `OpenAI` SDK 调用多家模型。

你将完成：

1. 列出 OpenRouter 上可用的模型（Models API）
2. 批量探测哪些模型能成功返回回复
3. 用不同模型扮演多个角色，跑一轮多角色对话（multi-agent conversation）

### 怎么跑

- 在 `.env` 中配置 `OPEN_ROUTER_API_KEY`（不要把密钥写进代码）
- 按顺序运行下方代码格；探测全部模型可能较慢，可先缩小范围再跑


In [ ]:
# ========== 列出 OpenRouter 可用模型 ==========
# 通过 Models API 拉取 openrouter 上当前可见的模型列表（需有效 Bearer token）
# With this api call we can get the list of models available in openrouter

# 导入 requests：用 HTTP GET 调用 REST 接口
import requests

# OpenRouter 官方模型列表端点（URL 勿改）
url = "https://openrouter.ai/api/v1/models"

# Authorization 头：Bearer 后接 API Token；这里用占位符 <token>，正式跑请换成真实密钥
headers = {"Authorization": "Bearer <token>"}

# 发起 GET 请求，拿到响应对象
response = requests.get(url, headers=headers)

# 把响应体解析为 JSON 并打印，便于在笔记本里查看模型清单
print(response.json())


In [ ]:
# ========== 拉取全部模型并逐个探测能否成功回复 ==========
# 通过这段代码，我们将从开放路由器中获取所有模型
# with this peice of code we are going to fetch all the models from open router
# 并尝试从每个模型中获取响应
# and try to get the response from each model
# 我们区分免费模式和付费模式（成功/失败结果会分别汇总）
# we are segregate with free vs paid models
# 保留这个以供参考
# keep this for reference

# 导入 requests：HTTP 调用 Models API
import requests
# 导入 os：从环境变量读取 API Key
import os
# 导入 json：本格虽未直接 dump，常与 API JSON 搭配使用
import json
# 从 dotenv 导入 load_dotenv：把 .env 密钥读进环境变量（Environment Variables）
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端：走 OpenRouter 的 OpenAI 兼容 Chat Completions
from openai import OpenAI

# 先建一个默认 OpenAI 客户端（本格后续主要用指向 OpenRouter 的 client）
openai = OpenAI()
# 加载 .env；override=True 表示用文件覆盖已有同名环境变量
load_dotenv(override=True)
# 读取 OpenRouter 密钥：环境变量名必须是 OPEN_ROUTER_API_KEY
api_key = os.getenv('OPEN_ROUTER_API_KEY')
# 指向 OpenRouter 的客户端：base_url 固定为官方 v1，api_key 用上面读到的密钥
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=api_key,
)

# 发给每个模型的第一条用户消息（prompt 原文保留，勿翻译）
message = "Hello, GPT! This is my first ever message to you! Hi!"

# Chat Completions 所需的 messages 列表：单轮 user
messages = [{"role": "user", "content": message}]
# 再次加载环境变量（原逻辑如此，保持不变）
# Load environment variables
load_dotenv(override=True)
# 再次读取密钥，供下面 Models API 的 Authorization 使用
api_key = os.getenv('OPEN_ROUTER_API_KEY')

# Models 列表接口 URL（勿改）
url = "https://openrouter.ai/api/v1/models"
# 带 Bearer 的请求头：用 f-string 把 api_key 拼进 Authorization
headers = {"Authorization": f"Bearer {api_key}"}

# 获取模型
# Fetch the models
response = requests.get(url, headers=headers)
# 解析 JSON：通常含 data 字段，每项是一个模型对象
data = response.json()

# 仅将 ID 提取到简单列表中
# Extract only the IDs into a simple list
model_ids = [model['id'] for model in data.get('data', [])]

# 创建最终对象（保留 model_ids 结构，便于对照）
# Create the final object
result = {"model_ids": model_ids}

# 收集每个模型探测结果（成功或失败）的列表
results = []

# 循环遍历模型 ID
# Loop through the model IDs
for model_id in model_ids:
    try:
        # 尝试调用模型
        # Attempt to call the model
        response = client.chat.completions.create(
            model=model_id, 
            messages=messages,
            timeout=15  # Recommended to avoid hanging on slow models
        )
        
        # 取出第一条 choice 的文本内容
        content = response.choices[0].message.content
        # 控制台标记：该 model_id 可用
        print(f"✅ Working: {model_id}")
        
        # 存储成功记录
        # Store success record
        results.append({
            "model_id": model_id,
            "status": "success",
            "content": content
        })
        
    except Exception as e:
        # 捕获任何错误（API 错误、超时等）
        # Catch any error (API errors, timeouts, etc.)
        print(f"❌ Error: {model_id} -> {str(e)}")
        
        # 存储错误记录
        # Store error record
        results.append({
            "model_id": model_id,
            "status": "error",
            "error_message": str(e)
        })

# 摘要清单
# Summary lists
# 筛出探测成功的模型 id
working_models = [r['model_id'] for r in results if r['status'] == 'success']
# 筛出探测失败的模型 id
failed_models = [r['model_id'] for r in results if r['status'] == 'error']

# 打印最终汇总标题与两类列表
print("\n--- FINAL SUMMARY ---")
print(f"Working Models: {working_models}")
print(f"Failed Models: {failed_models}")

# 详细视图（在 Jupyter 中交互）
# Detailed view (interactive in Jupyter)
from IPython.display import JSON


# 用 IPython JSON 组件展示完整 results，便于点开查看
JSON(results)


In [ ]:
# ========== 导入：后面对话实验要用的库 ==========

# 导入 os：读环境变量里的 API Key
import os
# 导入 requests：需要时可用 HTTP 直调接口
import requests
# 从 dotenv 导入 load_dotenv：加载 .env
from dotenv import load_dotenv
# 从 openai 导入 OpenAI：OpenRouter 兼容客户端
from openai import OpenAI
# 从 IPython.display 导入 Markdown、display：在笔记本里渲染富文本（本格先导入备用）
from IPython.display import Markdown, display


In [ ]:
# ========== 加载 OpenRouter 密钥 ==========

# 从 .env 读入环境变量；override=True 覆盖已有同名变量
load_dotenv(override=True)
# 读取 OPEN_ROUTER_API_KEY，赋给 openrouter_api_key 供下一格客户端使用
openrouter_api_key = os.getenv('OPEN_ROUTER_API_KEY')
# 打印密钥用于自检是否加载成功（注意：真实环境请勿把含密钥的输出提交到公开仓库）
print(openrouter_api_key)


In [ ]:
# ========== 多角色对话：三个角色、三个模型 ==========

# 创建指向 OpenRouter 的 OpenAI 兼容客户端
openrouter = OpenAI(base_url='https://openrouter.ai/api/v1', api_key=openrouter_api_key)
# Alex 使用的模型 id（OpenRouter 路由名，勿改）
gpt_model = "openai/gpt-oss-120b"
# Blake 使用的模型 id
deepseek_model = "deepseek/deepseek-v3.2"
# Charlie 使用的模型 id
gemma_model = "google/gemma-3-27b-it"

# Alex 的 system prompt：好辩、爱抬杠（原文保留，影响角色行为）
ALEX_SYSTEM = """You are Alex, a chatbot who is very argumentative;
you disagree with anything in the conversation and you challenge everything, in a snarky way.
You are in a conversation with Blake and Charlie."""

# Blake 的 system prompt：礼貌、爱找共同点
BLAKE_SYSTEM = """You are Blake, a very polite, courteous chatbot.
You try to agree with everything the other person says, or find common ground.
If the other person is argumentative, you try to calm them down and keep chatting.
You are in a conversation with Alex and Charlie."""

# Charlie 的 system prompt：务实、爱澄清与提下一步
CHARLIE_SYSTEM = """You are Charlie.
You are thoughtful, concise, and practical. You ask clarifying questions and propose next steps.
You are in a conversation with Alex and Blake."""


# 映射角色 -> 哪个模型将扮演它们
# Map personas -> which model will play them
CAST = {
    "Alex":   {"model": gpt_model,      "system": ALEX_SYSTEM},
    "Blake":  {"model": deepseek_model, "system": BLAKE_SYSTEM},
    "Charlie":{"model": gemma_model,    "system": CHARLIE_SYSTEM},
}

def format_transcript(history):
    # 历史记录：字典列表：{"speaker": "...", "text": "..."}
    # history: list of dicts: {"speaker": "...", "text": "..."}]
    # print("\n".join([f'{m["speaker"]}: {m["text"]}' 代表历史中的 m]))
    # print("\n".join([f'{m["speaker"]}: {m["text"]}' for m in history]))
    # 把整段对话拼成「说话人: 文本」的多行字符串，供下一轮模型当上下文
    return "\n".join([f'{m["speaker"]}: {m["text"]}' for m in history])

def ask_model(model, system_prompt, speaker_name, transcript):
    # 打印（模型）
    # print(model)
    # 构造 user prompt：告诉模型自己是谁、目前对话到哪、只输出下一句正文
    user_prompt = f"""You are {speaker_name}, in conversation with the others.
The conversation so far is as follows:
{transcript}

Now respond with what you would like to say next, as {speaker_name}.
Only output your message text (no speaker label)."""

    # 调用 Chat Completions：system + user；temperature=0.9 让回复更活泼
    resp = openrouter.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.9,
    )
    # 取第一条回复文本并去掉首尾空白
    return resp.choices[0].message.content.strip()

def run_conversation(seed_history, order=("Alex", "Blake", "Charlie"), rounds=3):
    # 复制种子历史，避免原地修改调用方传入的列表
    history = seed_history[:]
    # 外层：对话轮数；内层：按 order 依次让每个角色发言
    for _ in range(rounds):
        for speaker in order:
            # 把当前历史格式化成 transcript 字符串
            transcript = format_transcript(history)
            # 按角色名取出模型与 system prompt（嵌套字典）
            cfg = CAST[speaker] #nested dictionary check the case with speaker name
            # 请该角色的模型生成下一句
            text = ask_model(
                model=cfg["model"],
                system_prompt=cfg["system"],
                speaker_name=speaker,
                transcript=transcript,
            )
            # 把新发言追加进历史
            history.append({"speaker": speaker, "text": text})
    # 返回完整对话历史
    return history


# ---- 用你的起跑线来引导对话 ----
# ---- Seed the conversation using your starting lines ----
# 种子台词：三位角色各一句开场
history = [
    {"speaker": "Blake", "text": "Hi there"},
    {"speaker": "Charlie", "text": "Hi"},
    {"speaker": "Alex", "text": "what's up"},
]

# 从种子历史跑 4 轮（每人每轮各发言一次）
final_history = run_conversation(history, rounds=4)

# 打印完整 transcript，便于阅读整场对话
print(format_transcript(final_history))
